# 直方体展開図ジェネレーター (Cuboid Net Generator) - 完全スタンドアロン版

このノートブックは、他のローカルファイル（`fifth.py` や `make_cuboid.py`）に依存せず、**このファイル単体で Google Colab や Jupyter 上で完全に動作するように設計されています**。

直方体の寸法を指定するだけで、重なりのないユニークな展開図パターンを自動で列挙・計算し、グラフ同型判定（リボングラフ同型）を用いて重複を取り除き、可視化します。

## 1. ライブラリのインストールとセットアップ
まずは実行に必要な外部ライブラリをインストール・インポートします。

In [ ]:
# 必要なパッケージをインストール (Google Colabにはインストール済みのものが多いですが念のため実行します)
!pip install numpy matplotlib shapely networkx

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from shapely.geometry import Polygon
from typing import List, Tuple, Optional

## 2. コアアルゴリズムの実装
直方体を展開し、重なりや回転重複を排除するプログラムを読み込みます。このセルを実行するだけで、すべての関数が定義されます。

In [ ]:
def make_cuboid(w: float, h: float, d: float) -> Tuple[np.ndarray, List[List[int]]]:
    """与えられた 幅(w), 高さ(h), 奥行き(d) の直方体の頂点と面リストを生成する。"""
    vertices = np.array([
        [0, 0, 0],
        [w, 0, 0],
        [w, h, 0],
        [0, h, 0],
        [0, 0, d],
        [w, 0, d],
        [w, h, d],
        [0, h, d]
    ], dtype=float)
    faces = [
        [0, 3, 2, 1],  # 底面
        [4, 5, 6, 7],  # 上面
        [0, 1, 5, 4],  # 前面
        [1, 2, 6, 5],  # 右面
        [2, 3, 7, 6],  # 後面
        [3, 0, 4, 7]   # 左面
    ]
    return vertices, faces

def build_adjacency_graph(faces: List[List[int]]) -> List[Tuple[int, int]]:
    """面の隣接グラフの辺リストを構築する。"""
    edges = []
    n_faces = len(faces)
    for i in range(n_faces):
        for j in range(i + 1, n_faces):
            if len(set(faces[i]) & set(faces[j])) == 2:
                edges.append((i, j))
    return edges

def enumerate_spanning_trees(edges: List[Tuple[int, int]], num_nodes: int) -> List[List[Tuple[int, int]]]:
    """すべての全域木を列挙する。"""
    all_trees = []
    for edge_subset in itertools.combinations(edges, num_nodes - 1):
        g = nx.Graph()
        g.add_nodes_from(range(num_nodes))
        g.add_edges_from(edge_subset)
        if nx.is_tree(g):
            all_trees.append(list(edge_subset))
    return all_trees

def compute_normals(vertices: np.ndarray, faces: List[List[int]]) -> List[np.ndarray]:
    """各面の外向き法線ベクトルを計算する。"""
    normals = []
    for face in faces:
        p0 = vertices[face[0]]
        p1 = vertices[face[1]]
        p2 = vertices[face[2]]
        n = np.cross(p1 - p0, p2 - p1)
        norm = np.linalg.norm(n)
        if norm < 1e-9:
            raise ValueError("退化した面（面積ゼロ）が検出されました。")
        normals.append(n / norm)
    return normals

def layout_tree(vertices: np.ndarray, faces: List[List[int]], normals: List[np.ndarray], tree_edges: List[Tuple[int, int]]) -> List[Polygon]:
    """全域木に基づいて面を展開し、ShapelyのPolygonリスト（2D座標系）を返す。"""
    n_faces = len(faces)
    adj = [[] for _ in range(n_faces)]
    for u, v in tree_edges:
        adj[u].append(v)
        adj[v].append(u)
    
    m_matrices = [None] * n_faces
    m_matrices[0] = np.eye(4)
    queue = [0]
    visited = {0}
    
    while queue:
        u = queue.pop(0)
        for v in adj[u]:
            if v in visited:
                continue
            
            shared = set(faces[u]) & set(faces[v])
            n_u = len(faces[u])
            a_idx, b_idx = None, None
            for i in range(n_u):
                if faces[u][i] in shared and faces[u][(i + 1) % n_u] in shared:
                    a_idx = faces[u][i]
                    b_idx = faces[u][(i + 1) % n_u]
                    break
            if a_idx is None or b_idx is None:
                continue
                
            p_a = vertices[a_idx]
            p_b = vertices[b_idx]
            
            edge_vec = p_b - p_a
            k = edge_vec / np.linalg.norm(edge_vec)
            
            nu = normals[u]
            nv = normals[v]
            c = np.dot(nv, nu)
            s = np.dot(np.cross(k, nv), nu)
            
            kx, ky, kz = k
            v_1c = 1 - c
            r_rot = np.array([
                [kx * kx * v_1c + c, kx * ky * v_1c - kz * s, kx * kz * v_1c + ky * s, 0],
                [ky * kx * v_1c + kz * s, ky * ky * v_1c + c, ky * kz * v_1c - kx * s, 0],
                [kz * kx * v_1c - ky * s, kz * ky * v_1c + kx * s, kz * kz * v_1c + c, 0],
                [0, 0, 0, 1]
            ])
            
            t1 = np.eye(4)
            t1[0:3, 3] = -p_a
            t2 = np.eye(4)
            t2[0:3, 3] = p_a
            transform = t2 @ r_rot @ t1
            
            m_matrices[v] = m_matrices[u] @ transform
            visited.add(v)
            queue.append(v)
            
    origin = vertices[faces[0][0]]
    u_x = vertices[faces[0][1]] - origin
    u_x = u_x / np.linalg.norm(u_x)
    u_z = normals[0]
    u_y = np.cross(u_z, u_x)
    
    polygons = []
    for i in range(n_faces):
        pts_2d = []
        for v_idx in faces[i]:
            p3d_h = np.append(vertices[v_idx], 1.0)
            p_trans = (m_matrices[i] @ p3d_h)[:3]
            x2d = np.dot(p_trans - origin, u_x)
            y2d = np.dot(p_trans - origin, u_y)
            pts_2d.append((x2d, y2d))
        polygons.append(Polygon(pts_2d))
    return polygons

def is_valid_layout(polygons: List[Polygon]) -> bool:
    """展開図のポリゴン間に重なり（衝突）がないかを検証する。"""
    n_polys = len(polygons)
    for i in range(n_polys):
        for j in range(i + 1, n_polys):
            if polygons[i].intersection(polygons[j]).area > 1e-4:
                return False
    return True

def build_ribbon_graph(vertices: np.ndarray, faces: List[List[int]], tree_edges: List[Tuple[int, int]]) -> nx.DiGraph:
    """展開図の木構造から半辺グラフ (Ribbon Graph) を構築する。"""
    g = nx.DiGraph()
    for f_idx, face in enumerate(faces):
        n_v = len(face)
        for i in range(n_v):
            p1 = vertices[face[i]]
            p2 = vertices[face[(i + 1) % n_v]]
            length = np.linalg.norm(p2 - p1)
            node_id = (f_idx, i)
            g.add_node(node_id, length=round(length, 5))
        for i in range(n_v):
            g.add_edge((f_idx, i), (f_idx, (i + 1) % n_v), type="next")
            
    for u, v in tree_edges:
        shared = set(faces[u]) & set(faces[v])
        if len(shared) != 2:
            continue
        idx_u = None
        n_u = len(faces[u])
        for i in range(n_u):
            if faces[u][i] in shared and faces[u][(i + 1) % n_u] in shared:
                idx_u = i
                break
        idx_v = None
        n_v = len(faces[v])
        for i in range(n_v):
            if faces[v][i] in shared and faces[v][(i + 1) % n_v] in shared:
                idx_v = i
                break
                
        if idx_u is not None and idx_v is not None:
            u_start = faces[u][idx_u]
            u_end = faces[u][(idx_u + 1) % n_u]
            v_start = faces[v][idx_v]
            v_end = faces[v][(idx_v + 1) % n_v]
            flipped = (u_start == v_end) and (u_end == v_start)
            
            g.add_edge((u, idx_u), (v, idx_v), type="glue", flipped=flipped)
            g.add_edge((v, idx_v), (u, idx_u), type="glue", flipped=flipped)
    return g

def is_isomorphic_net(g1: nx.DiGraph, g2: nx.DiGraph) -> bool:
    """2つの半辺グラフが同型であるかを判定する。"""
    def node_match(n1, n2):
        return abs(n1["length"] - n2["length"]) < 1e-4
    def edge_match(e1, e2):
        if e1["type"] != e2["type"]:
            return False
        if e1["type"] == "glue":
            return e1.get("flipped") == e2.get("flipped")
        return True
    GM = nx.algorithms.isomorphism.DiGraphMatcher(g1, g2, node_match=node_match, edge_match=edge_match)
    return GM.is_isomorphic()

def reverse_ribbon_graph(g: nx.DiGraph) -> nx.DiGraph:
    """鏡像（裏返し）グラフを生成する。"""
    rev_g = nx.DiGraph()
    for node, data in g.nodes(data=True):
        rev_g.add_node(node, **data)
    for u, v, data in g.edges(data=True):
        if data["type"] == "next":
            rev_g.add_edge(v, u, **data)
        else:
            rev_g.add_edge(u, v, **data)
    return rev_g

def generate_unfolding_nets(vertices: np.ndarray, faces: List[List[int]]) -> List[List[Polygon]]:
    """一意な展開図に対応するポリゴンリストを生成する。"""
    edges = build_adjacency_graph(faces)
    all_trees = enumerate_spanning_trees(edges, len(faces))
    normals = compute_normals(vertices, faces)
    
    unique_trees = []
    unique_graphs = []
    unique_graphs_rev = []
    
    for tree in all_trees:
        try:
            polys = layout_tree(vertices, faces, normals, tree)
            if not is_valid_layout(polys):
                continue
            
            rg = build_ribbon_graph(vertices, faces, tree)
            is_duplicate = False
            for u_g, u_g_rev in zip(unique_graphs, unique_graphs_rev):
                if is_isomorphic_net(rg, u_g) or is_isomorphic_net(rg, u_g_rev):
                    is_duplicate = True
                    break
            if not is_duplicate:
                unique_graphs.append(rg)
                unique_graphs_rev.append(reverse_ribbon_graph(rg))
                unique_trees.append(tree)
        except Exception:
            continue
            
    valid_nets = []
    for tree in unique_trees:
        polys = layout_tree(vertices, faces, normals, tree)
        valid_nets.append(polys)
    return valid_nets

## 3. 寸法を指定して計算を実行する
直方体の **幅 (w)**、**高さ (h)**、**奥行き (d)** をミリメートルやセンチメートルなどの数値で指定して実行してください。

In [ ]:
# ここを好きな数値に変更して実行してください
w = 1.0  # 幅
h = 1.5  # 高さ
d = 2.0  # 奥行き

print(f"直方体寸法: 幅={w}, 高さ={h}, 奥行き={d}")

# 頂点と面データの作成
vertices, faces = make_cuboid(w, h, d)

# 展開図の計算
nets = generate_unfolding_nets(vertices, faces)

print(f"計算結果: ユニークな展開図は全部で {len(nets)} 通り見つかりました。")

## 4. 計算された展開図の可視化
計算結果から得られた展開図パターンを、グリッド状に並べてノートブック上に描画します。

In [ ]:
def show_nets_grid(polygons_list, max_display=12):
    display_count = min(len(polygons_list), max_display)
    if display_count == 0:
        print("表示する展開図はありません。")
        return
        
    cols = 3
    rows = (display_count + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    
    if display_count == 1:
        axes = np.array([[axes]])
    elif rows == 1:
        axes = np.array([axes])
        
    for idx in range(display_count):
        r, c = idx // cols, idx % cols
        ax = axes[r, c]
        polys = polygons_list[idx]
        
        all_x, all_y = [], []
        for face_id, poly in enumerate(polys):
            x, y = poly.exterior.xy
            all_x.extend(x)
            all_y.extend(y)
            ax.fill(x, y, alpha=0.3, fc="skyblue", ec="blue", lw=2)
            cx, cy = poly.centroid.x, poly.centroid.y
            ax.text(cx, cy, str(face_id), ha="center", va="center", fontsize=12, weight="bold")
            
        ax.set_aspect("equal", "box")
        ax.set_title(f"パターン {idx + 1}")
        margin = 0.5
        ax.set_xlim(min(all_x) - margin, max(all_x) + margin)
        ax.set_ylim(min(all_y) - margin, max(all_y) + margin)
        ax.grid(True, linestyle="--", alpha=0.5)
        
    for idx in range(display_count, rows * cols):
        r, c = idx // cols, idx % cols
        fig.delaxes(axes[r, c])
        
    plt.tight_layout()
    plt.show()

# 最初の12パターンを表示
show_nets_grid(nets, max_display=12)